In [1]:
import torch
from torch_geometric.explain import Explainer, GNNExplainer, ThresholdConfig

from models.node_pred_gnn import load_data
from utils.gcn_utils import get_device
from pathlib import Path

from configs.datasets.mimic import MimicConfig
from configs.formats.MEDSFormat import MEDSFormat
from configs.model import ModelConfig
from configs.experiment import ExperimentConfig

import torch.nn.functional as F
from torch.nn import Linear, PReLU
from torch_geometric.logging import log
from torch_geometric.nn import GCNConv, RGCNConv, GATConv
from torch.nn import Linear, PReLU, LayerNorm


# class RGCNNet(torch.nn.Module):
#     def __init__(
#         self,
#         embed_dim: int,
#         hidden_dim: int,
#         num_relations: int,
#         dropout: float,
#         num_classes: int,
#         include_text_features: bool,
#     ):
#         super().__init__()

#         self.include_text_features = include_text_features
#         self.dropout = dropout

#         # ----- Feature projections -----
#         self.numeric_projection = Linear(1, embed_dim)

#         if self.include_text_features:
#             self.text_projection = Linear(384, embed_dim)

#         self.input_activation = PReLU(embed_dim)

#         # ----- RGCN layers -----
#         self.conv1 = RGCNConv(embed_dim, hidden_dim, num_relations, num_bases=8)
#         self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations, num_bases=8)
#         self.conv3 = RGCNConv(hidden_dim, num_classes, num_relations, num_bases=8)

#         self.act1 = PReLU(hidden_dim)
#         self.act2 = PReLU(hidden_dim)

#     def forward(self, x, edge_index, data):
#         # ----- Initial feature fusion -----
#         node_features = self.numeric_projection(data.num_x)
#         node_features = self.input_activation(node_features)

#         if self.include_text_features:
#             text_features = self.text_projection(data.txt_x)
#             text_features = self.input_activation(text_features)
#             node_features = node_features + text_features

#         node_features = node_features + data.x

#         # ----- Graph convolutions -----
#         node_features = self.conv1(node_features, data.edge_index, data.edge_type)
#         node_features = self.act1(node_features)
#         node_features = F.dropout(node_features, p=self.dropout, training=self.training)

#         node_features = self.conv2(node_features, data.edge_index, data.edge_type)
#         node_features = self.act2(node_features)
#         node_features = F.dropout(node_features, p=self.dropout, training=self.training)

#         node_features = self.conv3(node_features, data.edge_index, data.edge_type)

#         return F.log_softmax(node_features, dim=1)


# class GCNet(torch.nn.Module):
#     def __init__(
#         self,
#         embed_dim: int,
#         hidden_dim: int,
#         num_relations: int,
#         dropout: float,
#         num_classes: int,
#         include_text_features: bool,
#     ):
#         super().__init__()

#         self.include_text_features = include_text_features
#         self.dropout = dropout

#         # ----- Feature projections -----
#         self.numeric_projection = Linear(1, embed_dim)

#         if self.include_text_features:
#             self.text_projection = Linear(384, embed_dim)

#         self.input_activation = PReLU(embed_dim)

#         # ----- RGCN layers -----
#         self.conv1 = GCNConv(embed_dim, hidden_dim)
#         self.conv2 = GCNConv(hidden_dim, hidden_dim)
#         self.conv3 = GCNConv(hidden_dim, num_classes)

#         self.act1 = PReLU(hidden_dim)
#         self.act2 = PReLU(hidden_dim)

#     def forward(self, x, edge_index, data):
#         # ----- Initial feature fusion -----
#         node_features = self.numeric_projection(data.num_x)
#         node_features = self.input_activation(node_features)

#         if self.include_text_features:
#             text_features = self.text_projection(data.txt_x)
#             text_features = self.input_activation(text_features)
#             node_features = node_features + text_features

#         node_features = node_features + x

#         # ----- Graph convolutions -----
#         node_features = self.conv1(node_features, edge_index)
#         node_features = self.act1(node_features)
#         node_features = F.dropout(node_features, p=self.dropout, training=self.training)

#         node_features = self.conv2(node_features, edge_index)
#         node_features = self.act2(node_features)
#         node_features = F.dropout(node_features, p=self.dropout, training=self.training)

#         node_features = self.conv3(node_features, edge_index)

#         return F.log_softmax(node_features, dim=1)


class GATNet(torch.nn.Module):
    def __init__(
        self,
        embed_dim: int,
        hidden_dim: int,
        dropout: float,
        num_classes: int,
        include_text_features: bool,
        num_relations = 0,
        num_heads: int = 4,
        text_dim: int = 384,
    ):
        super().__init__()

        self.include_text_features = include_text_features
        self.dropout = dropout
        self.num_heads = num_heads

        # ---- feature projections ----
        self.num_proj = Linear(1, embed_dim)

        self.node_proj = Linear(embed_dim, hidden_dim)

        self.input_activation = PReLU(embed_dim)

        # ---- GAT layers (NO edge types) ----
        self.conv1 = GATConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // num_heads,
            heads=num_heads,
            dropout=dropout,
            concat=True,
        )

        self.conv2 = GATConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim // num_heads,
            heads=num_heads,
            dropout=dropout,
            concat=True,
        )

        self.act1 = PReLU(hidden_dim)
        self.act2 = PReLU(hidden_dim)

        self.norm1 = LayerNorm(hidden_dim)
        self.norm2 = LayerNorm(hidden_dim)

        self.out = Linear(hidden_dim, 1)

    def forward(self, x, edge_index, data):
        # ----- Numeric features -----
        num_mask = data.num_mask.view(-1, 1)
        num_x = self.num_proj(data.num_x * num_mask)
        num_x = self.input_activation(num_x)

        h = num_x
        
        h = h + self.node_proj(data.x)

        # ---- GAT message passing (NO edge_attr) ----
        h = self.conv1(h, data.edge_index)
        h = self.norm1(h)
        h = self.act1(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, data.edge_index)
        h = self.norm2(h)
        h = self.act2(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.out(h)   # [N, 1]
        return h.squeeze(-1)

def build_id_to_uri(entity_df):
    entity_dict = dict(zip(entity_df[1], entity_df[0]))
    return {v: k for k, v in entity_dict.items()}


# ------------------------ Example Usage ------------------------ #


def load_model(a, b):
    # Load data
    data, patients, y = load_data(
        20000,
        a.embed_dim,
        b,
    )
    data.num_patients = 20000
    data.model_path = "results/meds/gat/first_24_in_hospital_mortality/0/0/model_weights.pth"

    # Get device and move data
    device = get_device()
    data = data.to(device)  # type: ignore

    model = GATNet(
        embed_dim=a.embed_dim,
        hidden_dim=a.hidden_dim,
        num_relations=data.num_relations,
        dropout=a.dropout,
        num_classes=2,
        include_text_features=True,
    )

    model.load_state_dict(torch.load(data.model_path, weights_only=True))
    model.to(device)
    model.eval()
    return model, data

/mnt/storage/conda/envs/gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
exp_cfg = ExperimentConfig(
    folds=5,
    dataset_samples=1,  # exp["num_of_samples"],
    time_option="TS",
    include_text=False,
    data_mode=MEDSFormat(),
    model_type="gat",
    # enrich_events = MIMIC_ENHANCER_DICT,
)
model_cfg = ModelConfig(lr=1e-4, hidden_dim=64, embed_dim=64, weight_decay=1e-4)

dataset_cfg = MimicConfig(
    source_dir=Path("/home/ubuntu/workspace/meds-to-owl-examples/exports"),
    num_patients=20000,
    task="first_24_in_hospital_mortality",
)

model, data = load_model(
    a=model_cfg,
    b=dataset_cfg.generate(0, exp_cfg),
)

In [3]:
import numpy as np

y_index = np.load("/home/ubuntu/workspace/patient-centric-GNNs/results/meds/gat/first_24_in_hospital_mortality/0/0/y_index.npy")
y = np.load("/home/ubuntu/workspace/patient-centric-GNNs/results/meds/gat/first_24_in_hospital_mortality/0/0/y_true.npy")
y_pred = np.load(Path("/home/ubuntu/workspace/patient-centric-GNNs/results/meds/gat/first_24_in_hospital_mortality/0/0/y_pred.npy"))

In [4]:
print(y_index[10], y[10], y_pred[10])
print(y_index[0], y[0], y_pred[0])
print(y_index[1], y[1], y_pred[1])

#true_positive = 1972553  
true_negative = 706214  
#false_positive = 2523862

2466894 0.0 1
706214 0.0 0
2462873 1.0 0


In [5]:
explainer = Explainer(
    model=model,
    algorithm=GNNExplainer(epochs=200),
    explanation_type="model",
    node_mask_type=None,
    edge_mask_type="object",
    model_config=dict(
        mode="binary_classification",
        task_level="node",
        return_type="raw",
    ),
    threshold_config=ThresholdConfig(threshold_type="topk", value=100),
)
#print(f"Generated explanations in {explanation.available_explanations}")

In [6]:
import pandas as pd

entity_df = pd.read_csv(
    "/home/ubuntu/workspace/patient-centric-GNNs/processed_data/first_24_in_hospital_mortality/0/meds_TS_entities_20000.tsv",
    sep="\t",
    header=None,
)
entity_dict = dict(zip(entity_df[1], entity_df[0]))
id_to_entity = dict(zip(entity_df[0], entity_df[1]))


In [7]:
indexes = list({k: v.removeprefix("https://teamheka.github.io") for k, v in id_to_entity.items()}.values())

In [10]:
node_index = 706214
explanation = explainer(data.x, data.edge_index, index=node_index, data=data)
print(f'Generated explanations in {explanation.available_explanations}')

Generated explanations in ['edge_mask']


In [12]:
explanation.threshold(threshold_type="topk", value=10).visualize_graph("subgraph.png", backend="graphviz", node_labels=indexes)

#explanation.visualize_graph("subgraph.png", backend="graphviz", node_labels=indexes)
#explanation.visualize_feature_importance("feature_importance.png", top_k=15)

In [13]:
from torch_geometric.explain.metric import (
   fidelity,
   characterization_score, 
   fidelity_curve_auc,
   unfaithfulness
)

is_valid = explanation.validate()

print(is_valid)

# Fidelity
fid_pos, fid_neg = fidelity(
   explainer=explainer,
   explanation=explanation#.threshold(threshold_type="topk", value=3000)
)

print(f"Fidelity Score: {fid_pos}, {fid_neg}")


#Characterization score
# char_score = characterization_score(
#     fid_pos,
#     fid_neg,
#     pos_weight=0.7,    # Higher weight
#     neg_weight=0.3     # Lower weight        
# )

# # Fidelity curve AUC
# pos_fidelity = torch.tensor([0.9, 0.8, 0.7, 0.6, 0.5]) 
# neg_fidelity = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5]) 

# # x-axis values (e.g., thresholds or steps), sorted in ascending order
# x = torch.tensor([0.1, 0.2, 0.3, 0.4, 0.5])

# # Call the fidelity_curve_auc function
# auc = fidelity_curve_auc(pos_fidelity, neg_fidelity, x)


# Print results
# print(f"Characterization Score: {char_score}")
# print("Fidelity Curve AUC:", auc.item())

True
Fidelity Score: 0.0, 0.0
